# Anomaly Detection in Time-Series Data

This work was done for predictive maintenance at a steel pipe manufacturer.

<img src="../images/three-roll-mills.png" alt="FACTORY" style="width: 12cm; height: auto;" />


The function of the mill is to smooth the surface of the steel pipe.
In the factory, there are generally five mills aligned together to smooth the surface of the
steel pipe. It looks like this:

<img src="../images/FACTORY.png" alt="FACTORY" style="width: 12cm; height: auto;" />

During production, the surface of the roll cracks and then breaks. The cracks affect the
quality of the steel pipe, and the breakage of the roll will cause unplanned breakdown. The
factory must find cracks in advance and then replace the roll.

<img src="../images/crack.png" alt="FACTORY" style="width: 12cm; height: auto;" />

## 1. Anomaly Detection Using an Autoencoder
The system collects around 140 sensor values from motor speed, reference speed, torque, current, roll speed, and gap.
These values are fed into an autoencoder neural network.

<img src="../images/autoencoder.png" alt="autoencoder" style="width: 12cm; height: auto;" />

The system has an inner dimension of 8, and input and output dimensions of 140. After training on days of no-anomally in the sytem, we take the 
standard deviation of the error between the input and output values and its mean as the threshold. Any value exceeds that, is considered to be anomally.

```python
threshold = np.mean(train_loss) + np.std(train_loss)
```

<img src="../images/threshold.png" alt="FACTORY" style="width: 12cm; height: auto;" />

I applied PCA to reduce the latent dimension to 2 and observed that anomalies become clearly separable in this low-dimensional space.

<img src="../images/pca.png" alt="pca" style="width: 12cm; height: auto;" />

This suggests that a simple clustering approach could distinguish normal and anomalous behavior.
However, the factory chose to keep the standard method because the clustering-based approach was less aligned with the mechanical interpretation and required frequent tuning.

## 2. Vibration Analysis
### 2.1 Why Spectral Analysis?
The autoencoder solution worked well for roll cracks, but for bearing faults, changes in current, voltage, or rotating speed/torque may not appear until late. Vibration analysis can detect bearing cracks earlier.

<img src="../images/bearing.png" alt="FACTORY" style="width: 12cm; height: auto;" />

Rotating machinery emits vibration signals that contain rich information about mechanical health. As components degrade:
- **Frequency content changes** — new harmonics appear, amplitudes shift
- **Transient events** — impacts, cracks, spalls generate broadband bursts
- **Noise floor rises** — general wear increases random vibration

**Goal:** Detect these changes early, before catastrophic failure.


### Three approaches to spectral analysis:


| Method | Resolution | Time-Freq | Speed | Best For |
|--------|-----------|-----------|-------|----------|
| FFT | Fixed (df = 1/T) | No | Fast | Stationary signals |
| MESA | Adaptive, very high | No | Medium | Short data, sharp peaks |
| Wavelet | Variable | Yes | Medium | Non-stationary, transients |


### 2.2 Maximum Entropy Spectral Analysis (MESA)

#### The Problem with FFT

The FFT assumes the signal is periodic or zero-padded outside the observed window — this introduces spectral leakage and limits frequency resolution.

For short vibration bursts or rapidly changing conditions, this is a real constraint.

#### MESA: The Key Insight

[MESA (Burg, 1967)](https://link.springer.com/rwe/10.1007/978-3-030-85040-1_197) takes a different approach: instead of assuming zeros outside the observed data, it **extends the autocorrelation sequence by maximizing entropy** — making the minimal assumptions about unknown information.

**In plain terms:** MESA fits an autoregressive (AR) model to the data, then computes the spectrum from the model parameters. This gives much higher frequency resolution from short data segments.

### How it works:
1. The signal x[t] is modeled as an AR process
2. Burg's method estimates coefficients by minimizing forward/backward prediction error
3. The power spectral density is computed from the AR coefficients


## 2.3 What Is a Wavelet?

> A wavelet is a "brief oscillation" — a function that starts at zero, rises, oscillates, and returns to zero.

### The Morlet Wavelet (most common for CWT-based PSD)

![Morlet wavelet — real and imaginary parts](https://upload.wikimedia.org/wikipedia/commons/thumb/0/0a/MorletWaveletMathematica.svg/500px-MorletWaveletMathematica.svg.png)

A complex exponential (carrier) multiplied by a Gaussian envelope:

$$\psi(t) = \pi^{-1/4} e^{i \omega_0 t} e^{-t^2/2}$$

**Why Morlet?** Good time-frequency localization — the Gaussian envelope minimizes the Heisenberg uncertainty product.

### Other Common Wavelet Families

| Wavelet | Shape | Best For |
|---------|-------|----------|
| **Morlet** | Complex sinusoid × Gaussian | CWT, time-frequency analysis, PSD |
| **Mexican Hat** | Second derivative of Gaussian | Peak detection, zero-crossings |
| **Meyer** | Smooth, orthogonal | Signal denoising |
| **Daubechies** | Compact support, orthogonal | DWT, compression |

![Mexican hat wavelet](https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/MexicanHatMathematica.svg/500px-MexicanHatMathematica.svg.png)  |  ![Meyer wavelet](https://upload.wikimedia.org/wikipedia/commons/thumb/e/eb/MeyerMathematica.svg/500px-MeyerMathematica.svg.png)
:-------------------------:|:-------------------------:
**Mexican Hat** (real)    | **Meyer** (real)

![Daubechies 4 — scaling function (φ) and wavelet function (ψ)](https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Daubechies4-functions.svg/250px-Daubechies4-functions.svg.png)

*Daubechies 4: scaling function (left) and wavelet function (right) — compact support enables efficient DWT.*

---

### 2.3.1 The Continuous Wavelet Transform (CWT)

The CWT correlates a signal $x(t)$ with scaled and shifted copies of the mother wavelet $\psi(t)$:

$$W_x(a, b) = \frac{1}{\sqrt{a}} \int_{-\infty}^{\infty} x(t) \cdot \psi^*\left(\frac{t-b}{a}\right) dt$$

Where:
- **Scale $a$** → inversely related to frequency (small $a$ = high freq, large $a$ = low freq)
- **Translation $b$** → time shift
- $W_x(a, b)$ → wavelet **coefficients** = power at each time-frequency point


The **scalogram** is the wavelet analogue of the spectrogram:

$$S(a, b) = |W_x(a, b)|^2$$



In [ ]:
# --- Setup ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import rc
import pandas as pd
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import matplotlib as mpl
mpl.rcParams["animation.embed_limit"] = 40  # MB (default is 20)
# MESA
from memspectrum import MESA


# Wavelet
import pycwt as wavelet
from pycwt.helpers import find
import pywt

sns.set(style='whitegrid', palette='deep', font_scale=1.2)
rc('animation', html='html5')
print("All imports OK")

## 3. MESA, FFT and Wavelet Demo: Synthetic Signal

Let's start with a simple known signal: a 50 Hz sine wave + noise. We'll compare FFT vs MESA resolution.

In [ ]:
# --- MESA Demo: Clean sine + noise ---
N, fs = 200, 1000  # samples, sampling frequency (Hz)
dt = 1 / fs
t = np.arange(0, N) * dt

# Signal: 50 Hz + 120 Hz + noise
f1, f2 = 50, 120
signal = (np.sin(2 * np.pi * f1 * t) + 
          0.5 * np.sin(2 * np.pi * f2 * t) +
          0.4 * np.random.randn(N))

# --- FFT-based PSD ---
from scipy import signal as scipy_signal
f_fft, psd_fft = scipy_signal.periodogram(signal, fs, scaling='density')

# --- MESA-based PSD ---
M = MESA()
M.solve(signal)
f_mesa, psd_mesa = M.spectrum(dt)

# --- Wavelet "PSD" (global wavelet power spectrum) ---
mother = wavelet.Morlet(6)
dj = 1 / 12
s0 = 2 * dt
J = 7 / dj
signal_norm = (signal - np.mean(signal)) / np.std(signal)
wave, scales, freqs, coi, _, _ = wavelet.cwt(signal_norm, dt, dj, s0, J, mother)
wavelet_power = np.abs(wave) ** 2
global_wavelet_power = np.mean(wavelet_power, axis=1)

# --- Plot ---
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(t, signal, color='k', linewidth=0.8)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Signal: {f1} Hz + {f2} Hz + noise')

axes[1].semilogy(f_fft, psd_fft, linewidth=1)
axes[1].set_xlim([0, 200])
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('PSD')
axes[1].set_ylim([1e-6, 1])
axes[1].set_title('FFT PSD')
axes[1].axvline(f1, color='r', linestyle='--', alpha=0.4)
axes[1].axvline(f2, color='r', linestyle='--', alpha=0.4)

axes[2].semilogy(f_mesa, psd_mesa, linewidth=1, color='green')
axes[2].set_xlim([0, 200])
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel('PSD')
axes[2].set_title('MESA Spectrum')
axes[2].axvline(f1, color='r', linestyle='--', alpha=0.4)
axes[2].axvline(f2, color='r', linestyle='--', alpha=0.4)

axes[3].semilogy(freqs, global_wavelet_power, linewidth=1, color='purple')
axes[3].set_xlim([0, 200])
axes[3].set_xlabel('Frequency (Hz)')
axes[3].set_ylabel('Power')
axes[3].set_title('Wavelet Global Power')
axes[3].axvline(f1, color='r', linestyle='--', alpha=0.4)
axes[3].axvline(f2, color='r', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

print("Notice: FFT, MESA, and wavelet global power all identify dominant frequencies; MESA is sharper, while wavelets retain time-frequency structure.")

## 4. Application: Bearing Vibration Analysis

### The NASA IMS Bearing Dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy.special import entr
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from datetime import datetime
print(np.version.version)

# Dataset Description
Four bearings were installed on a shaft. The rotation speed was kept constant at 2000 RPM by an AC motor coupled to the shaft via rub belts. A radial load of 6000 lbs is applied onto the shaft and bearing by a spring mechanism. All bearings are force lubricated.
Rexnord ZA-2115 double row bearings were installed on the shaft as shown in Figure 1. PCB 353B33 High Sensitivity Quartz ICP accelerometers were installed on the bearing housing (two accelerometers for each bearing [x- and y-axes] for data set 1, one accelerometer for each bearing for data sets 2 and 3). Sensor placement is also shown in Figure 1. All failures occurred after exceeding designed life time of the bearing which is more than 100 million revolutions.

## Data Structure

Three (3) data sets are included in the data packet (IMS-Rexnord Bearing Data.zip). Each data set describes a test-to-failure experiment. Each data set consists of individual files that are 1-second vibration signal snapshots recorded at specific intervals. Each file consists of 20,480 points with the sampling rate set at 20 kHz. The file name indicates when the data was collected. Each record (row) in the data file is a data point. Data collection was facilitated by NI DAQ Card 6062E. Larger intervals of time stamps (showed in file names) indicate resumption of the experiment in the next working day.

## Set No. 1

| Index                    | Description                                                                                                                   |
|--------------------------|-------------------------------------------------------------------------------------------------------------------------------|
| Recording Duration:      | October 22, 2003 12:06:24 to November 25, 2003 23:39:56                                                                       |
| No. of Files:            | 2,156                                                                                                                         |
| No. of Channels:         | 8                                                                                                                             |
| Channel Arrangement:     | Bearing 1 – Ch 1&2; Bearing 2 – Ch 3&4; Bearing 3 – Ch 5&6; Bearing 4 – Ch 7&8.                                               |
| File Recording Interval: | Every 10 minutes (except the first 43 files were taken every 5 minutes)                                                       |
| File Format:             | ASCII                                                                                                                         |
| Description:             | At the end of the test-to-failure experiment, inner race defect occurred in bearing 3 and roller element defect in bearing 4. |

## Set No. 2

| Index                    | Description                                                                             |
|--------------------------|-----------------------------------------------------------------------------------------|
| Recording Duration:      | February 12, 2004 10:32:39 to February 19, 2004 06:22:39                                |
| No. of Files:            | 984                                                                                     |
| No. of Channels:         | 4                                                                                       |
| Channel Arrangement:     | Bearing 1 – Ch 1; Bearing2 – Ch 2; Bearing3 – Ch3; Bearing 4 – Ch 4.                    |
| File Recording Interval: | Every 10 minutes                                                                        |
| File Format:             | ASCII                                                                                   |
| Description:             | At the end of the test-to-failure experiment, outer race failure occurred in bearing 1. |

## Set No. 3

| Index                    | Description                                                                             |
|--------------------------|-----------------------------------------------------------------------------------------|
| Recording Duration:      | March 4, 2004 09:27:46 to April 4, 2004 19:01:57                                        |
| No. of Files:            | 4,448                                                                                   |
| No. of Channels:         | 4                                                                                       |
| Channel Arrangement:     | Bearing 1 – Ch 1; Bearing2 – Ch 2; Bearing3 – Ch3; Bearing 4 – Ch 4.                    |
| File Recording Interval: | Every 10 minutes                                                                        |
| File Format:             | ASCII                                                                                   |
| Description:             | At the end of the test-to-failure experiment, outer race failure occurred in bearing 3. |

## Vibration Data for experimentation
[Another NASA Bearing Dataset](https://ti.arc.nasa.gov/tech/dash/groups/pcoe/prognostic-data-repository/) (1Gb)


[Explanation to vibration data](https://www.youtube.com/embed/Vj1xmze3GlE)


<iframe width="720" height="405"
        src="https://www.youtube.com/embed/Vj1xmze3GlE"
        title="YouTube video" frameborder="0"
        allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share"
        referrerpolicy="strict-origin-when-cross-origin"
        allowfullscreen></iframe>

In [ ]:
dataset_path_1st = '../bearing-dataset/1st_test/1st_test/'
# Test for the first file
dataset = pd.read_csv((dataset_path_1st+'2003.10.22.12.06.24'), sep='\t')
dataset.columns = ['Bearing_1_ch1', 'Bearing_1_ch2','Bearing_2_ch3','Bearing_2_ch4','Bearing_3_ch5','Bearing_3_ch6', 'Bearing_4_ch7','Bearing_4_ch8']
dataset.head()

In [ ]:
fig, ax1 = plt.subplots(1,1, figsize=(16, 5))
dataset[['Bearing_1_ch1']].plot(figsize=(18,6), ax=ax1)
dataset[['Bearing_1_ch2']].plot(figsize=(18,6), ax=ax1)
plt.tight_layout()

In [ ]:
fig, ax1 = plt.subplots(1,1, figsize=(16, 5))
dataset.plot(figsize=(18,6), ax=ax1)
plt.tight_layout()

In [ ]:
def merge_data(dataset_path, id_set=None):
    data = []
    for filename in tqdm(os.listdir(dataset_path)):
        dataset=pd.read_csv(os.path.join(dataset_path, filename), sep='\t')
        if id_set == 1:
            dataset.columns =  ['B1_a','B1_b','B2_a','B2_b','B3_a','B3_b','B4_a','B4_b']
        else:
            dataset.columns =  ['B1','B2','B3','B4']
            
        dataset['Timestamp'] = datetime.strptime(filename, '%Y.%m.%d.%H.%M.%S')
        data.append(dataset)
        
    return pd.concat(data, axis=0)
all_data_df = merge_data(dataset_path_1st, 1)

In [ ]:
N, dt = len(all_data_df)//len(all_data_df.Timestamp.unique()), 1/20000  #Number of samples and sampling interval
time = np.arange(0, N) * dt
data = all_data_df[all_data_df.Timestamp== datetime.strptime('2003.11.24.06.41.24', '%Y.%m.%d.%H.%M.%S')]['B4_a']
# solving for MESA
M = MESA()
M.solve(data)
# The spectrum can be computed on sampling frequencies (automatically generated) or on some given interval
frequencies, spectrum  = M.spectrum(dt)  #Computes on sampling frequencies
fig, ax = plt.subplots(1,sharex=True)
spectrum[np.argmax(frequencies)]
data_df = pd.DataFrame()
data_df['Freq'] = frequencies
data_df['PSD']=spectrum
data_df.set_index('Freq', inplace=True)
data_df.sort_index(inplace=True)
data_df.loc[10:, 'PSD'].plot(ax=ax)
#ax[0].plot(spectrum,frequencies,  label='Default')
ax.set_xscale('log')
ax.set_ylabel('PSD')

In [ ]:
dates = sorted(list(all_data_df.Timestamp.unique()))
def process_date(date, all_data_df, channel):
    global N, dt
    data = all_data_df.loc[all_data_df.Timestamp==date, channel]
    
    M = MESA()
    M.solve(data)
    # The spectrum can be computed on sampling frequencies (automatically generated) or on some given interval
    freq, spec = M.spectrum(dt)
    return freq, spec, data  #Computes on sampling frequencies

In [ ]:
# Set up formatting for the movie files
Writer = animation.writers['ffmpeg']
writer = Writer(fps=15, metadata=dict(artist='Me'), bitrate=1800)
channel = 'B4_a'

fig, ax = plt.subplots(1,2,figsize=(10, 5))

date = dates[0]

# ax[0].xticks(rotation=70)
# ax[1].xticks(rotation=70)

ax[0].set_ylim([-1,1])
ax[1].set_ylim([0,2.5e-5])

ax[0].set_ylabel(channel)
ax[1].set_ylabel('PSD')
ax[0].set_xlabel('Time (sec)')
ax[1].set_xlabel('Frequencies (Hz)')
ax[1].set_xscale('log')
freq, spectrum, signal = process_date(date, all_data_df, channel)
line1,  = ax[0].plot(signal.index*dt, signal)

data_df = pd.DataFrame()
data_df['Freq'] = freq
data_df['PSD']=spectrum
data_df = data_df.loc[data_df.Freq>10]


line2,  = ax[1].plot(data_df.Freq, data_df.PSD)


title = ax[0].text(0.5,.75, "", bbox={'facecolor':'w', 'alpha':0.5, 'pad':5},
                 ha="center")
ax[0].grid()
ax[1].grid()
plt.tight_layout()

plt.close()
def animate(date):
    freq, spectrum, signal = process_date(date, all_data_df, channel)
    data_df = pd.DataFrame()
    data_df['Freq'] = freq
    data_df['PSD']=spectrum
    data_df = data_df.loc[data_df.Freq>10]
    line1.set_ydata(signal.values)
    line2.set_ydata(data_df.PSD)
    #sc.set_offsets( y)
    
    title.set_text(u"Date = {}".format(date.strftime("%Y-%m-%d")))
    #time.sleep(0.5)
    return None

line_ani = animation.FuncAnimation(fig, animate, dates, interval=20)
line_ani

In [ ]:
from IPython.display import Video
video = Video('../downlo.mp4')
display(video)

## Converting Acceleration to Velocity and Displacement

Accelerometers measure acceleration ($m/s^2$). For vibration analysis, we often want:
- **Velocity** ($m/s$): Related to kinetic energy and bearing wear
- **Displacement** ($m$): Related to shaft deflection and clearance

### Frequency-Domain Integration

The conversion uses integration in the frequency domain:

$$a(t) \xrightarrow{FFT} A(f) \xrightarrow{\div (j \cdot 2\pi f)} V(f) \xrightarrow{IFFT} v(t)$$

$$v(t) \xrightarrow{FFT} V(f) \xrightarrow{\div (j \cdot 2\pi f)} D(f) \xrightarrow{IFFT} d(t)$$

Where $j = \sqrt{-1}$ and $f$ is frequency in Hz. The division by $j\cdot 2\pi f$ acts as an integrator in the frequency domain — equivalent to $\int a(t)\,dt$ in the time domain.

**Reference:** [endaq — Top Vibration Metrics to Monitor](https://blog.endaq.com/top-vibration-metrics-to-monitor-how-to-calculate-them)

In [ ]:
def wavelet_denoise(signal, wavelet='db4', level=None, threshold_mode='soft'):
    """DWT soft-threshold denoising (VisuShrink universal threshold).

    Parameters
    ----------
    signal : array-like
        Input time series
    wavelet : str
        Wavelet family (default: Daubechies-4, matches DWT discussion above)
    level : int, optional
        Decomposition level; auto-selected if None
    threshold_mode : str
        'soft' or 'hard' thresholding

    Returns
    -------
    ndarray
        Denoised signal (same length as input)
    """
    signal = np.array(signal, dtype=float, copy=True)
    n = len(signal)
    max_level = pywt.dwt_max_level(n, pywt.Wavelet(wavelet).dec_len)
    level = level or max(1, min(max_level, int(np.log2(n)) - 2))

    coeffs = pywt.wavedec(signal, wavelet, level=level)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(n))

    denoised_coeffs = [coeffs[0]]
    denoised_coeffs.extend(
        pywt.threshold(c, threshold, mode=threshold_mode) for c in coeffs[1:]
    )
    return pywt.waverec(denoised_coeffs, wavelet)[:n]


def _freq_domain_integrate(signal, dt):
    """Integrate a time series once in the frequency domain."""
    signal = np.array(signal, dtype=float, copy=True)
    n = len(signal)
    freqs = np.fft.fftfreq(n, dt)
    signal_fft = np.fft.fft(signal - np.mean(signal))

    integrated_fft = np.zeros_like(signal_fft, dtype=complex)
    nonzero = freqs != 0
    integrated_fft[nonzero] = signal_fft[nonzero] / (2j * np.pi * freqs[nonzero])

    return np.fft.ifft(integrated_fft).real


def accel_to_velocity(accel, dt, denoise=True, wavelet='db4'):
    """Convert acceleration to velocity via wavelet denoising + frequency-domain integration.

    Parameters
    ----------
    accel : array-like
        Acceleration time series (m/s^2)
    dt : float
        Sampling interval (seconds)
    denoise : bool
        Apply DWT denoising before integration (default True)
    wavelet : str
        Wavelet family for denoising

    Returns
    -------
    velocity : ndarray
        Velocity time series (m/s)
    """
    accel = np.asarray(accel, dtype=float)
    if denoise:
        accel = wavelet_denoise(accel, wavelet=wavelet)
    return _freq_domain_integrate(accel, dt)


def velocity_to_displacement(vel, dt, denoise=True, wavelet='db4'):
    """Convert velocity to displacement via wavelet denoising + frequency-domain integration.

    Parameters
    ----------
    vel : array-like
        Velocity time series (m/s)
    dt : float
        Sampling interval (seconds)
    denoise : bool
        Apply DWT denoising before integration (default True)
    wavelet : str
        Wavelet family for denoising

    Returns
    -------
    displacement : ndarray
        Displacement time series (m)
    """
    vel = np.asarray(vel, dtype=float)
    if denoise:
        vel = wavelet_denoise(vel, wavelet=wavelet)
    return _freq_domain_integrate(vel, dt)


def accel_to_displacement(accel, dt, denoise=True, wavelet='db4'):
    """Convert acceleration to displacement (double integration with wavelet pre-processing)."""
    vel = accel_to_velocity(accel, dt, denoise=denoise, wavelet=wavelet)
    disp = velocity_to_displacement(vel, dt, denoise=denoise, wavelet=wavelet)
    return vel, disp

### Demo: Converting Bearing Vibration Data

Let's apply the conversion to our bearing dataset. We'll take one channel's acceleration data and compute velocity and displacement.

In [ ]:
def process_vib_date(date, all_data_df, channel):
    global N, dt
    data = all_data_df.loc[all_data_df.Timestamp==date, channel]

    # Convert
    velocity, displacement = accel_to_displacement(data, dt)
    
    return data, velocity, displacement



In [ ]:
# Set up formatting for the movie files
Writer = animation.writers['ffmpeg']
writer = Writer(fps=15, metadata=dict(artist='Me'), bitrate=1800)
channel = 'B4_a'

fig, ax = plt.subplots(3,1,figsize=(10,10), sharex=True)

date = dates[0]

# ax[0].xticks(rotation=70)
# ax[1].xticks(rotation=70)
accel, vel, disp = process_vib_date(date, all_data_df, channel)

ax[0].set_ylim([-1,1])
ax[1].set_ylim([-0.0005, 0.0005])
ax[2].set_ylim([-2e-4, 2e-4] )

ax[0].set_ylabel('Acceleration (m/s²)')
ax[1].set_ylabel('Velocity (m/s)')
ax[2].set_ylabel('Displacement (m)')
ax[2].set_xlabel('Time (sec)')
line0, = ax[0].plot(accel.index*dt, accel)
line1, = ax[1].plot(accel.index*dt, vel)
line2, = ax[2].plot(accel.index*dt, disp)

title = ax[0].text(0.5,.75, "", bbox={'facecolor':'w', 'alpha':0.5, 'pad':5},
                 ha="center")
plt.tight_layout()
plt.close()
def animate(date):
    accel, vel, disp = process_vib_date(date, all_data_df, channel)
    x = accel.index*dt
    line0.set_ydata(accel)
    line1.set_ydata(vel)
    line2.set_ydata(disp)
    #sc.set_offsets( y)
    
    title.set_text(u"Date = {}".format(date.strftime("%Y-%m-%d")))
    #time.sleep(0.5)
    return None

line_ani = animation.FuncAnimation(fig, animate, dates, interval=20)
line_ani

In [ ]:
video = Video('../download-velocity-displacement.mp4')
display(video)

In [ ]:
bearing_channels = ['B1_a', 'B2_a', 'B3_a', 'B4_a']
records = []
for date in tqdm(dates, desc='RMS per date'):
    row = {'Timestamp': date}
    for ch in bearing_channels:
        signal = all_data_df.loc[all_data_df.Timestamp == date, ch].to_numpy()
        _, disp = accel_to_displacement(signal, dt)
        row[ch] = float(np.sqrt(np.mean(disp ** 2)))
    records.append(row)
rms_df = pd.DataFrame(records).set_index('Timestamp').sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
for ch in bearing_channels:
    ax.plot(rms_df.index, rms_df[ch], label=ch, linewidth=1.2)
ax.set_yscale('log')
ax.set_ylabel('Displacement RMS (m)')
ax.set_xlabel('Date')
ax.set_title('Bearing displacement RMS over time — NASA IMS 1st test')